In [1]:
file_name = 'recording_20251029_140939.mp4'
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
file_path = '/content/drive/MyDrive/srl_data/' + file_name

In [3]:
!pip install transformers torch huggingface_hub

In [4]:
# from huggingface_hub import notebook_login

# notebook_login()


In [5]:
import transformers
import torch

In [6]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
model_name = "nvidia/Cosmos-Reason2-2B"
model = transformers.Qwen3VLForConditionalGeneration.from_pretrained(
    model_name, dtype=torch.float16, device_map="auto", attn_implementation="sdpa"
)
processor: transformers.Qwen3VLProcessor = (
    transformers.AutoProcessor.from_pretrained(model_name)
)



#

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/626 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [9]:
video_messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a helpful assistant."}],
    },
    {"role": "user", "content": [
            {
                "type": "video",
                "video": "/content/drive/MyDrive/srl_data/recording_20251029_140939.mp4",
                "fps": 4,
            },
            {"type": "text", "text": (
                    "Return the normalized coordinates for the center of mass of the satellite at the beginning and end of the video as [x1, y1] and [x2, y2] respectively. The top-left is (0,0) "+
                    "and the bottom-right is (1000,1000). "+
                    "And return the orientation of the satellite in Euler rotation angles at the beginning and end of the "+
                    "video respectively as [i1, j1, k1] and [i2, j2, k2]. "+
                    "Also return the normalized velocities in the horizontal and vertical axis "+
                    "at the beginning and end of the video as [h1x, h1y] and [h2x, h2y] for horizontal, and [V1x, V1y] and [V2x, V2y] for vertical respectively "+
                      "with negative values implying leftward or downward velocities in the axis respectively. "+
                     "Put your final answer in a json of the values in the schema: {\"center_of_mass_beginning\": [x1,y1]\n \"center_of_mass_end\": [x2,y2]\n \"orientation_beginning\": [i1,j1,k1]\n \"orientation_end\":[i2,j2,k2]\n \"velocity_horizontal_beginning\": [h1x,h1y]\n \"velocity_horizontal_end\": [h2x,h2y]\n \"velocity_vertical_beginning\": [V1x,V1y]\n \"velocity_vertical_end\": [V2x,V2y]}"+
                     "Answer the question using the following format:\n\n<think>\nYour reasoning.\n</think>\n\nWrite your final answer immediately after the </think> tag."
                )
            },
        ]
    },
]

In [12]:
#  Process inputs
inputs = processor.apply_chat_template(
    video_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
    fps=4,
)
inputs = inputs.to(model.device)

# Run inference
generated_ids = model.generate(**inputs, max_new_tokens=4096)
generated_ids_trimmed = [
    out_ids[len(in_ids) :]
    for in_ids, out_ids in zip(inputs.input_ids, generated_ids, strict=False)
]
output_text = processor.batch_decode(
    generated_ids_trimmed,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)


In [13]:
output_text

['<think>\nOkay, let\'s try to figure out the answer here. The user is asking for the normalized coordinates of the center of mass of the satellite at the beginning and end of the video, along with the orientation and velocity. \n\nFirst, I need to look at the video. The video shows a satellite with a central cube-like structure and two rectangular panels on either side. The camera is static, focusing on the satellite. The key points shown are the gradual reveal of the satellite, the center of mass, and the orientation.\n\nAt the beginning, the video starts with a dark screen, and the satellite is not fully visible. The video shows that the satellite\'s center of mass is at the center, and the orientation is shown as a cube with panels. The initial orientation is likely a cube with the panels on the sides. The camera is static, so the position of the center of mass remains the same throughout the video.\n\nThe center of mass is a fixed point in the satellite\'s structure. Since the sat